# Olira SDK — Document resource ingestion

Upload clinical PDFs/images so Olira can OCR them and emit EventLog rows.

Two paths:

| Path | API | Scope | When to use |
| --- | --- | --- | --- |
| **A — Live upload** | `upload_document()` → poll `DocumentResource` | `sdk:event-log` | One-off / ongoing document intake (no human confirm) |
| **B — Historical package** | `create_ingestion_job(records=…, documents=[IngestDocument(…)])` | `sdk:historical-ingest` | Bulk backfill of PDFs (± patients/logs) through the historical pipeline |

Caller chooses target types — the platform does not infer them from the file:

- `unstructured_report` → requires `document_type` (catalog enum, e.g. `pathology_report`)
- `clinical_note` → requires `note_type` + `source` (e.g. `progress_note` + `manual_entry`)

**Setup:** copy `.env.example` → `.env`, set `OLIRA_API_KEY`, then `uv sync` in this folder.

In [ ]:
import os
import time
import uuid
from datetime import UTC, datetime
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

from olira import (
    DEFAULT_BASE_URL,
    CreatePatientRequest,
    ExternalIdentifier,
    IngestDocument,
    IngestLogSpec,
    IngestRecord,
    OliraClient,
    OliraEnv,
)
from olira.documents import DocumentLogType, DocumentStatus

API_KEY = os.environ["OLIRA_API_KEY"]
BASE_URL = os.environ.get("OLIRA_BASE_URL", DEFAULT_BASE_URL)

client = OliraClient(
    api_key=API_KEY,
    base_url=BASE_URL,
    environment=OliraEnv.DEVELOPMENT if "localhost" in BASE_URL else OliraEnv.PRODUCTION,
    async_flush=False,
)
print(f"base_url={BASE_URL}")

## Sample PDF

Set `DOCUMENT_PATH` in `.env` to a real PDF/PNG/JPEG, or this cell writes a tiny stub PDF for the demo.

In [ ]:
DOCUMENT_PATH = os.environ.get("DOCUMENT_PATH")

if DOCUMENT_PATH:
    pdf_path = Path(DOCUMENT_PATH).expanduser().resolve()
    if not pdf_path.is_file():
        raise FileNotFoundError(f"DOCUMENT_PATH not found: {pdf_path}")
else:
    pdf_path = Path.cwd() / "sample_note.pdf"
    # Minimal PDF so the upload APIs accept the bytes (OCR on stubs may fail — use a real scan for end-to-end OCR).
    pdf_path.write_bytes(
        b"%PDF-1.4\n"
        b"1 0 obj<< /Type /Catalog /Pages 2 0 R >>endobj\n"
        b"2 0 obj<< /Type /Pages /Kids [3 0 R] /Count 1 >>endobj\n"
        b"3 0 obj<< /Type /Page /Parent 2 0 R /MediaBox [0 0 300 144] "
        b"/Contents 4 0 R /Resources<< /Font<< /F1 5 0 R >> >> >>endobj\n"
        b"4 0 obj<< /Length 44 >>stream\n"
        b"BT /F1 12 Tf 72 72 Td (Olira demo note) Tj ET\n"
        b"endstream\nendobj\n"
        b"5 0 obj<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>endobj\n"
        b"xref\n0 6\n0000000000 65535 f \n"
        b"trailer<< /Size 6 /Root 1 0 R >>\nstartxref\n0\n%%EOF\n"
    )

print(f"Using document: {pdf_path} ({pdf_path.stat().st_size} bytes)")

## Create a demo patient

Requires `api:manage-patients`.

In [ ]:
run_id = uuid.uuid4().hex[:8]

patient = client.create_patient(
    first_name="Doc",
    last_name=f"Resource{run_id}",
    timezone="America/New_York",
    external_identifiers=[ExternalIdentifier(system="demo", value=f"DOC-RES-{run_id}")],
)
print(f"Patient created: {patient.id}")

## Path A — Live document upload (`DocumentResource`)

Flow: upload-url → presigned PUT (+ `Content-Type`) → commit → OCR → EventLog.

Statuses: `pending_upload` → `uploaded` → `ocr_running` → `ocr_complete` → `log_emitted` (or `ocr_failed`).
`ocr_complete` is **non-terminal** — wait until `log_emitted` / `ocr_failed`.

Requires `sdk:event-log`.

In [ ]:
handle = client.upload_document(
    patient_id=patient.id,
    path=pdf_path,
    log_type=DocumentLogType.UNSTRUCTURED_REPORT,
    document_type="pathology_report",  # catalog enum for unstructured_report
    timestamp=datetime(2025, 6, 15, 14, 30, tzinfo=UTC),
    idempotency_key=f"demo-live-doc-{run_id}",
)
print(f"Uploaded document_id={handle.document_id} status={handle.document.status}")

# Block until OCR finishes and the EventLog is emitted (or fails).
# For a stub PDF this may land in ocr_failed — swap DOCUMENT_PATH for a real scan to exercise OCR.
doc = handle.wait(timeout_s=600.0, poll_interval_s=3.0)
print(f"Final status={doc.status}")
print(f"  event_log_id={doc.event_log_id}")
print(f"  ocr_page_count={doc.ocr_page_count} ocr_confidence={doc.ocr_confidence}")
if doc.error:
    print(f"  error={doc.error}")

assert doc.status in (DocumentStatus.LOG_EMITTED, DocumentStatus.OCR_FAILED)

### Optional — clinical note target

Same live path, different labels (`note_type` + `source` instead of `document_type`).

In [ ]:
note_handle = client.upload_document(
    patient_id=patient.id,
    path=pdf_path,
    log_type=DocumentLogType.CLINICAL_NOTE,
    note_type="progress_note",
    source="manual_entry",  # manual_entry | ehr_integration | hie_integration
    timestamp=datetime(2025, 6, 16, 9, 0, tzinfo=UTC),
    idempotency_key=f"demo-live-note-{run_id}",
    wait=True,  # same as handle.wait()
    wait_timeout_s=600.0,
)
note = note_handle.document
print(f"document_id={note.document_id} status={note.status} event_log_id={note.event_log_id}")

## Path B — Historical document package

Bulk path for onboarding backfills. The SDK:

1. `POST /v1/ingestion/jobs:begin`
2. PUTs each binary (+ matching `Content-Type`)
3. Builds `manifest.jsonl` (`type=document` rows, plus any patient/log records)
4. Creates the job

Pipeline (with `require_confirmation=True`):

```
QUEUED → … → AWAITING_CONFIRMATION  (Phase 1 — cheap)
  → confirm → EXTRACTING (OCR) → REPLAYING → … → COMPLETED
```

Pass binaries via `documents=` — do **not** put `IngestRecord.document(...)` in `records=`.

Requires `sdk:historical-ingest`.

In [ ]:
def poll_until(job_id, target_statuses, interval=5, timeout=600):
    deadline = time.time() + timeout
    while time.time() < deadline:
        job = client.get_ingestion_job(job_id=job_id)
        eta = f"  ETA ~{job.estimated_seconds_remaining}s" if job.estimated_seconds_remaining else ""
        print(f"  [{job.status}] {job.progress_pct:.0f}%  {job.stage}{eta}")
        if job.status in target_statuses:
            return job
        time.sleep(interval)
    raise TimeoutError(f"Job {job_id} did not reach {target_statuses} within {timeout}s")


ext_id = f"PKG-{run_id}"
records = [
    IngestRecord.patient(
        CreatePatientRequest(
            first_name="Pkg",
            last_name=f"Demo{run_id}",
            timezone="UTC",
            external_identifiers=[ExternalIdentifier(system="demo", value=ext_id)],
        )
    ),
    IngestRecord.log(
        IngestLogSpec(
            event_type="moods_report",
            patient_id=ext_id,
            timestamp="2025-05-01T09:00:00Z",
            payload={"moods": [{"mood": "hopeful", "intensity": 6}], "source": "checkin"},
            idempotency_key=f"pkg-mood-{run_id}",
        )
    ),
]

documents = [
    IngestDocument(
        path=str(pdf_path),
        patient_id=ext_id,  # external_identifier value — resolved with the patient row above
        log_type="unstructured_report",
        document_type="radiology_imaging",
        timestamp="2025-05-02T10:00:00Z",
        ref_id="d1",
        idempotency_key=f"pkg-doc-{run_id}-d1",
    ),
]

job = client.create_ingestion_job(
    records=records,
    documents=documents,
    idempotency_key=f"demo-doc-package-{run_id}",
    require_confirmation=True,
)
print(f"Job created: {job.job_id} (status={job.status})")

In [ ]:
job = poll_until(job.job_id, {"awaiting_confirmation", "failed", "completed"}, interval=5)

if job.status == "awaiting_confirmation":
    print("Phase 1 review:")
    print(f"  patients_processed     : {job.patients_processed}")
    print(f"  logs_processed         : {job.logs_processed}  (failed: {job.logs_failed})")
    print(f"  documents_total        : {job.documents_total}")
    print(f"  documents_registered   : {job.documents_registered}")
    if job.error_summary:
        for err in job.error_summary[:5]:
            print(f"  error line {err.line}: [{err.code}] {err.message}")

    job = client.confirm_ingestion_job(job_id=job.job_id)
    print("\nConfirmed — Phase 2 (OCR + replay) started…")
    job = poll_until(
        job.job_id,
        {"completed", "completed_with_errors", "failed"},
        interval=15,
        timeout=1800,
    )

print(f"\nFinal status: {job.status}")
print(
    f"  documents: total={job.documents_total} "
    f"ocr_ok={job.documents_ocr_succeeded} ocr_failed={job.documents_ocr_failed}"
)

## Cleanup

Demo-only — remove these deletes when adapting the notebook for a real integration.

In [ ]:
client.delete_patient(patient_id=patient.id)
print(f"Deleted live-path patient {patient.id}")

# Package path created its own patient via the ingestion job.
pkg = client.list_patients(external_system="demo", external_value=ext_id)
if pkg.patients:
    client.delete_patient(patient_id=pkg.patients[0].id)
    print(f"Deleted package-path patient {pkg.patients[0].id}")

client.close()
print("Done.")